# Data Exam (Elite) — CoderPad Practice Problems

If you can solve these cold in 30 min, you're overqualified. These combine multiple systems, require careful algorithmic thinking, and have tricky edge cases. All specs and formulas are still provided — the difficulty is execution under pressure.

Only attempt these after MEDIUM and HARD feel comfortable.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import numpy as np
from typing import Any, Iterator
from collections import defaultdict, Counter, OrderedDict
import math
import heapq

---

## Problem 1: Full Embedding Pipeline

### Scenario

You're building a complete data curation system for a video dataset. Given pre-computed embeddings for each item, you need to: store and manage them, detect near-duplicates, and then select a maximally diverse subset.

### Formulas

**Cosine similarity:** `cos(a, b) = (a · b) / (||a|| * ||b||)`

**Connected components:** Two items are in the same cluster if there exists a path of near-duplicate pairs between them. Use BFS or union-find.

**Farthest-first traversal (diversity selection):**
1. Start with the item that has the highest L2 norm (arbitrary tiebreaker)
2. At each step, add the item whose minimum cosine distance to any already-selected item is maximized
3. Cosine distance = 1 - cosine_similarity

### Requirements

**`EmbeddingStore`:**
- `__init__(self)` — empty store
- `add(self, item_id: str, embedding: torch.Tensor, metadata: dict | None = None)` — add an item. Raise ValueError if item_id already exists.
- `remove(self, item_id: str)` — remove an item. Raise KeyError if not found.
- `update(self, item_id: str, embedding: torch.Tensor | None = None, metadata: dict | None = None)` — update embedding and/or metadata. Raise KeyError if not found.
- `get(self, item_id: str) -> dict` — return `{"embedding": tensor, "metadata": dict}`
- `get_all_embeddings(self) -> tuple[list[str], torch.Tensor]` — return (list of IDs, stacked embeddings matrix (N, D))
- `__len__(self)` — number of stored items

**`DuplicateDetector`:**
- `__init__(self, store: EmbeddingStore, threshold: float)`
- `find_clusters(self) -> list[set[str]]` — find connected components of items with pairwise cosine similarity >= threshold. Return only clusters with 2+ items.

**`DiversitySelector`:**
- `__init__(self, store: EmbeddingStore)`
- `select(self, candidate_ids: list[str], target_count: int) -> list[str]` — farthest-first traversal on the given candidate IDs. Return up to target_count IDs. If target_count >= len(candidate_ids), return all.

**`run_pipeline(store: EmbeddingStore, dup_threshold: float, target_count: int) -> dict`:**
- Detect duplicate clusters
- For each cluster, keep only the item with the highest L2 norm embedding (arbitrary tiebreaker for ties)
- From the remaining (kept + non-duplicated) items, run diversity selection to pick target_count items
- Return `{"selected_ids": list[str], "duplicate_clusters": list[set[str]], "removed_as_duplicates": list[str], "total_original": int, "total_after_dedup": int, "total_selected": int}`

In [ ]:
class EmbeddingStore:
    # YOUR CODE HERE
    pass


class DuplicateDetector:
    # YOUR CODE HERE
    pass


class DiversitySelector:
    # YOUR CODE HERE
    pass


def run_pipeline(store: EmbeddingStore, dup_threshold: float, target_count: int) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
torch.manual_seed(42)

# === EmbeddingStore basic operations ===
store = EmbeddingStore()
store.add("a", torch.randn(16), {"source": "web"})
store.add("b", torch.randn(16), {"source": "scraped"})
assert len(store) == 2

# Duplicate ID should raise
try:
    store.add("a", torch.randn(16))
    assert False, "Should raise ValueError for duplicate ID"
except ValueError:
    pass

# Get
item = store.get("a")
assert item["embedding"].shape == (16,)
assert item["metadata"]["source"] == "web"

# Update
new_emb = torch.ones(16)
store.update("a", embedding=new_emb)
assert torch.allclose(store.get("a")["embedding"], new_emb)
store.update("a", metadata={"source": "updated"})
assert store.get("a")["metadata"]["source"] == "updated"

# Remove
store.remove("b")
assert len(store) == 1
try:
    store.remove("nonexistent")
    assert False, "Should raise KeyError"
except KeyError:
    pass

# get_all_embeddings
store.add("b", torch.randn(16))
store.add("c", torch.randn(16))
ids, emb_matrix = store.get_all_embeddings()
assert len(ids) == 3
assert emb_matrix.shape == (3, 16)

# === DuplicateDetector ===
torch.manual_seed(42)
dup_store = EmbeddingStore()
base_a = F.normalize(torch.randn(32), dim=0)
base_b = F.normalize(torch.randn(32), dim=0)

# Cluster A: items 0, 1, 2 are near-duplicates
dup_store.add("item_0", base_a + 0.01 * torch.randn(32))
dup_store.add("item_1", base_a + 0.01 * torch.randn(32))
dup_store.add("item_2", base_a + 0.01 * torch.randn(32))
# Cluster B: items 3, 4
dup_store.add("item_3", base_b + 0.01 * torch.randn(32))
dup_store.add("item_4", base_b + 0.01 * torch.randn(32))
# Unique item
dup_store.add("item_5", F.normalize(torch.randn(32), dim=0))

detector = DuplicateDetector(dup_store, threshold=0.95)
clusters = detector.find_clusters()
assert len(clusters) == 2, f"Expected 2 clusters, got {len(clusters)}"

# Find which cluster contains item_0
cluster_a = [c for c in clusters if "item_0" in c][0]
cluster_b = [c for c in clusters if "item_3" in c][0]
assert cluster_a == {"item_0", "item_1", "item_2"}
assert cluster_b == {"item_3", "item_4"}
assert not any("item_5" in c for c in clusters), "Unique item should not be in any cluster"

# === DiversitySelector ===
torch.manual_seed(42)
div_store = EmbeddingStore()
# Place items at known positions for predictable diversity selection
div_store.add("north", torch.tensor([0.0, 1.0, 0.0, 0.0]))
div_store.add("south", torch.tensor([0.0, -1.0, 0.0, 0.0]))
div_store.add("east", torch.tensor([1.0, 0.0, 0.0, 0.0]))
div_store.add("west", torch.tensor([-1.0, 0.0, 0.0, 0.0]))
div_store.add("near_north", torch.tensor([0.0, 0.99, 0.0, 0.0]))

selector = DiversitySelector(div_store)
selected = selector.select(["north", "south", "east", "west", "near_north"], target_count=3)
assert len(selected) == 3
# near_north is very close to north, so it should NOT be among the first 3 selected
assert "near_north" not in selected, f"near_north should not be selected early: {selected}"
# All 4 cardinal directions should be preferred over near_north
selected_4 = selector.select(["north", "south", "east", "west", "near_north"], target_count=4)
assert "near_north" not in selected_4, f"near_north should still not be in top 4: {selected_4}"

# target_count >= candidates returns all
selected_all = selector.select(["north", "south"], target_count=10)
assert len(selected_all) == 2

# === Full pipeline integration ===
result = run_pipeline(dup_store, dup_threshold=0.95, target_count=3)
assert result["total_original"] == 6
assert len(result["duplicate_clusters"]) == 2
assert len(result["removed_as_duplicates"]) > 0
assert result["total_after_dedup"] == result["total_original"] - len(result["removed_as_duplicates"])
assert result["total_selected"] == 3
assert len(result["selected_ids"]) == 3
# Selected IDs should not include any removed duplicates
removed_set = set(result["removed_as_duplicates"])
for sid in result["selected_ids"]:
    assert sid not in removed_set, f"Selected ID {sid} was removed as duplicate"

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Dynamic Batch Collator

### Scenario

Your multi-modal training pipeline has records containing images of different sizes, variable-length token sequences, and variable numbers of bounding boxes. You need a collator that handles all of these simultaneously, plus a smart batcher that minimizes padding waste.

### Requirements

**`dynamic_collate(batch: list[dict]) -> dict`:**

Each record in the batch has:
- `"image"`: float tensor of shape (C, H, W) — variable H, W across records
- `"tokens"`: 1D int64 tensor — variable length
- `"bboxes"`: float tensor of shape (num_boxes, 4) — variable num_boxes
- `"label"`: int scalar

The collator must:
1. **Images:** Find max H and max W in the batch. Resize all images to (C, max_H, max_W) using `F.interpolate` (mode='bilinear', align_corners=False). Input to interpolate must be 4D: add batch dim, interpolate, squeeze.
2. **Tokens:** Pad to max length with 0. Create a `"token_mask"` (1 for real tokens, 0 for padding).
3. **Bboxes:** Pad to max num_boxes with zeros. Create a `"bbox_mask"` (1 for real boxes, 0 for padding).
4. **Labels:** Stack into a 1D tensor.
5. **Padding stats:** Compute `"padding_stats"` dict:
   - `"image_pixels_wasted"`: total interpolated pixels minus total original pixels (sum across batch of max_H*max_W - orig_H*orig_W)
   - `"token_cells_wasted"`: total padding tokens (sum of max_len - orig_len per item)
   - `"bbox_cells_wasted"`: total padding boxes (sum of max_boxes - orig_boxes per item)

Return: `{"images": (B,C,max_H,max_W), "tokens": (B,max_len), "token_mask": (B,max_len), "bboxes": (B,max_boxes,4), "bbox_mask": (B,max_boxes), "labels": (B,), "padding_stats": dict}`

**`SmartBatcher`:**
- `__init__(self, records: list[dict], batch_size: int)` — records as described above
- Pre-sort records by image area (H * W) and group into contiguous batches
- `__iter__(self) -> Iterator[list[dict]]` — yield batches (lists of records)
- `__len__(self)` — number of batches

The key insight: batches of similar-sized images waste fewer pixels on resizing than random batches.

In [ ]:
def dynamic_collate(batch: list[dict]) -> dict:
    # YOUR CODE HERE
    pass


class SmartBatcher:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 2 ---
torch.manual_seed(42)

# Create a batch with variable sizes
batch = [
    {"image": torch.randn(3, 32, 48), "tokens": torch.tensor([1, 2, 3], dtype=torch.int64),
     "bboxes": torch.randn(2, 4), "label": 0},
    {"image": torch.randn(3, 64, 32), "tokens": torch.tensor([4, 5, 6, 7, 8], dtype=torch.int64),
     "bboxes": torch.randn(5, 4), "label": 1},
    {"image": torch.randn(3, 48, 64), "tokens": torch.tensor([9], dtype=torch.int64),
     "bboxes": torch.randn(1, 4), "label": 2},
]

collated = dynamic_collate(batch)

# Shape checks
assert collated["images"].shape == (3, 3, 64, 64), f"Expected (3,3,64,64), got {collated['images'].shape}"
assert collated["tokens"].shape == (3, 5), f"Expected (3,5), got {collated['tokens'].shape}"
assert collated["token_mask"].shape == (3, 5)
assert collated["bboxes"].shape == (3, 5, 4), f"Expected (3,5,4), got {collated['bboxes'].shape}"
assert collated["bbox_mask"].shape == (3, 5)
assert collated["labels"].shape == (3,)

# Dtype checks
assert collated["images"].dtype == torch.float32
assert collated["tokens"].dtype == torch.int64
assert collated["token_mask"].dtype == torch.int64 or collated["token_mask"].dtype == torch.bool
assert collated["labels"].dtype == torch.int64

# Token mask correctness
assert collated["token_mask"][0].sum().item() == 3  # first record has 3 tokens
assert collated["token_mask"][1].sum().item() == 5  # second has 5
assert collated["token_mask"][2].sum().item() == 1  # third has 1

# Bbox mask correctness
assert collated["bbox_mask"][0].sum().item() == 2
assert collated["bbox_mask"][1].sum().item() == 5
assert collated["bbox_mask"][2].sum().item() == 1

# Labels
assert collated["labels"].tolist() == [0, 1, 2]

# Token padding — padded positions should be 0
assert collated["tokens"][0, 3:].sum().item() == 0  # last 2 are padding
assert collated["tokens"][2, 1:].sum().item() == 0  # last 4 are padding

# Padding stats
stats = collated["padding_stats"]
assert "image_pixels_wasted" in stats
assert "token_cells_wasted" in stats
assert "bbox_cells_wasted" in stats
# Token waste: (5-3) + (5-5) + (5-1) = 2 + 0 + 4 = 6
assert stats["token_cells_wasted"] == 6, f"Expected 6, got {stats['token_cells_wasted']}"
# Bbox waste: (5-2) + (5-5) + (5-1) = 3 + 0 + 4 = 7
assert stats["bbox_cells_wasted"] == 7, f"Expected 7, got {stats['bbox_cells_wasted']}"
# Image pixel waste: 3 * (64*64) - (32*48 + 64*32 + 48*64) = 12288 - (1536+2048+3072) = 12288 - 6656 = 5632
assert stats["image_pixels_wasted"] == 5632, f"Expected 5632, got {stats['image_pixels_wasted']}"

# === SmartBatcher ===
# Create records with varying image sizes
np.random.seed(42)
records = []
for i in range(40):
    h = np.random.choice([32, 64, 128, 256])
    w = np.random.choice([32, 64, 128, 256])
    records.append({
        "image": torch.randn(3, h, w),
        "tokens": torch.randint(1, 100, (np.random.randint(1, 10),), dtype=torch.int64),
        "bboxes": torch.randn(np.random.randint(1, 5), 4),
        "label": np.random.randint(0, 3),
    })

smart = SmartBatcher(records, batch_size=8)
assert len(smart) == 5  # 40 / 8

smart_batches = list(smart)
assert len(smart_batches) == 5
assert all(len(b) == 8 for b in smart_batches)

# Collate all smart batches and random batches, compare waste
import random
random.seed(42)
shuffled = records.copy()
random.shuffle(shuffled)
random_batches = [shuffled[i:i+8] for i in range(0, 40, 8)]

smart_waste = sum(dynamic_collate(b)["padding_stats"]["image_pixels_wasted"] for b in smart_batches)
random_waste = sum(dynamic_collate(b)["padding_stats"]["image_pixels_wasted"] for b in random_batches)

print(f"Smart waste: {smart_waste}, Random waste: {random_waste}")
assert smart_waste <= random_waste, (
    f"Smart batching should reduce waste: smart={smart_waste}, random={random_waste}"
)

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Annotation Reconciliation System

### Scenario

Multiple annotators label the same items (e.g., video quality ratings). Annotators disagree. You need a system that reconciles disagreements using multiple strategies, and reports annotator quality.

### Formulas

**Majority vote:** The label with the most votes wins. Ties broken by alphabetical order of label.

**Weighted majority:** Each vote is weighted by its confidence. Sum weights per label. Highest total weight wins. Same tiebreaker.

**Dawid-Skene (one EM step):**
1. Start with current consensus (majority vote) as "ground truth" estimate for each item.
2. **E-step — estimate annotator accuracy:** For each annotator, accuracy = (number of annotations matching current consensus) / (total annotations by that annotator).
3. **M-step — re-weight votes:** For each item, for each label, sum the accuracies of annotators who voted for that label. The label with the highest sum wins.
4. Return the M-step result as the new consensus.

**Pairwise agreement:** For an item with K annotators, agreement = (number of annotator pairs that gave the same label) / (total annotator pairs = K*(K-1)/2). If K < 2, agreement = 1.0.

### Requirements

**`AnnotationStore`:**
- `__init__(self)`
- `add(self, item_id: str, annotator_id: str, label: str, confidence: float)` — add an annotation. If the same annotator already labeled the same item, overwrite.
- `get_annotations(self, item_id: str) -> list[dict]` — return list of `{"annotator_id": str, "label": str, "confidence": float}` for that item
- `get_items(self) -> list[str]` — list of all item IDs
- `get_annotators(self) -> list[str]` — list of all annotator IDs
- `reconcile(self, item_id: str, method: str) -> str` — resolve label using specified method ("majority", "weighted_majority", "dawid_skene_step")
- `compute_agreement(self, item_id: str) -> float` — pairwise agreement rate
- `annotator_quality_report(self) -> dict` — for each annotator: `{"accuracy": float, "num_annotations": int, "avg_confidence": float}`. Accuracy is fraction of annotations matching majority-vote consensus.

**Note on Dawid-Skene:** The `reconcile` method with `"dawid_skene_step"` needs ALL items' annotations to estimate annotator accuracies, not just the queried item.

In [ ]:
class AnnotationStore:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 3 ---

astore = AnnotationStore()

# Item 1: clear majority for "good"
astore.add("item_1", "ann_A", "good", 0.9)
astore.add("item_1", "ann_B", "good", 0.8)
astore.add("item_1", "ann_C", "bad", 0.7)

# Item 2: split vote, confidence breaks it for weighted
astore.add("item_2", "ann_A", "good", 0.6)
astore.add("item_2", "ann_B", "bad", 0.95)
astore.add("item_2", "ann_C", "bad", 0.3)

# Item 3: unanimous
astore.add("item_3", "ann_A", "good", 0.9)
astore.add("item_3", "ann_B", "good", 0.85)
astore.add("item_3", "ann_C", "good", 0.7)

# === Basic storage ===
assert len(astore.get_annotations("item_1")) == 3
assert set(astore.get_items()) == {"item_1", "item_2", "item_3"}
assert set(astore.get_annotators()) == {"ann_A", "ann_B", "ann_C"}

# Overwrite: same annotator + same item should overwrite
astore.add("item_1", "ann_C", "mediocre", 0.5)
annots_1 = astore.get_annotations("item_1")
ann_c_labels = [a["label"] for a in annots_1 if a["annotator_id"] == "ann_C"]
assert ann_c_labels == ["mediocre"], f"Overwrite failed: {ann_c_labels}"
# Restore for subsequent tests
astore.add("item_1", "ann_C", "bad", 0.7)

# === Majority vote ===
assert astore.reconcile("item_1", "majority") == "good"  # 2 good vs 1 bad
assert astore.reconcile("item_2", "majority") == "bad"   # 1 good vs 2 bad
assert astore.reconcile("item_3", "majority") == "good"  # unanimous

# === Weighted majority ===
assert astore.reconcile("item_1", "weighted_majority") == "good"  # good: 0.9+0.8=1.7 vs bad: 0.7
# Item 2: good=0.6, bad=0.95+0.3=1.25 → bad wins
assert astore.reconcile("item_2", "weighted_majority") == "bad"

# === Dawid-Skene step ===
# Consensus (majority): item_1=good, item_2=bad, item_3=good
# ann_A: item1=good(match), item2=good(no match), item3=good(match) → accuracy=2/3
# ann_B: item1=good(match), item2=bad(match), item3=good(match) → accuracy=3/3=1.0
# ann_C: item1=bad(no match), item2=bad(match), item3=good(match) → accuracy=2/3
# Item 1 M-step: good votes from A(2/3)+B(1.0)=5/3, bad votes from C(2/3)
ds_result = astore.reconcile("item_1", "dawid_skene_step")
assert ds_result == "good", f"Dawid-Skene for item_1: expected 'good', got '{ds_result}'"

# === Pairwise agreement ===
# Item 1: A=good, B=good, C=bad → pairs: (A,B)=agree, (A,C)=disagree, (B,C)=disagree → 1/3
agree_1 = astore.compute_agreement("item_1")
assert abs(agree_1 - 1/3) < 1e-5, f"Expected 1/3, got {agree_1}"

# Item 3: all agree → 3/3 = 1.0
agree_3 = astore.compute_agreement("item_3")
assert abs(agree_3 - 1.0) < 1e-5, f"Expected 1.0, got {agree_3}"

# === Annotator quality report ===
report = astore.annotator_quality_report()
assert set(report.keys()) == {"ann_A", "ann_B", "ann_C"}
assert report["ann_B"]["num_annotations"] == 3
# ann_B matches consensus on all 3 items
assert abs(report["ann_B"]["accuracy"] - 1.0) < 1e-5
# ann_A: matches on item_1(good=good) and item_3(good=good), not item_2(good!=bad) → 2/3
assert abs(report["ann_A"]["accuracy"] - 2/3) < 1e-5, f"Expected 2/3, got {report['ann_A']['accuracy']}"
# ann_A avg confidence = (0.9 + 0.6 + 0.9) / 3 = 0.8
assert abs(report["ann_A"]["avg_confidence"] - 0.8) < 1e-5

# === Edge case: single annotator ===
solo_store = AnnotationStore()
solo_store.add("x", "solo", "cat", 0.9)
assert solo_store.reconcile("x", "majority") == "cat"
assert solo_store.compute_agreement("x") == 1.0  # K < 2 → 1.0

# === Edge case: tie in majority → alphabetical ===
tie_store = AnnotationStore()
tie_store.add("t", "a1", "zebra", 0.5)
tie_store.add("t", "a2", "apple", 0.5)
assert tie_store.reconcile("t", "majority") == "apple", "Tie should break alphabetically"

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Curriculum Training Scheduler

### Scenario

Research shows that training on easy examples first and gradually introducing harder ones (curriculum learning) improves convergence. Build a scheduler that manages this.

### Formulas

**Pacing function:** At training progress `p` (0.0 to 1.0), include data up to difficulty percentile:
```
max_percentile(p) = min(1.0, start_percentile + p * (1.0 - start_percentile))
```
Where `start_percentile` is a hyperparameter (e.g., 0.2 means start with the easiest 20%).

So at `p=0`: include data up to percentile `start_percentile` (e.g., easiest 20%).
At `p=1`: include everything (`percentile=1.0`).

### Requirements

**`DifficultyScorer`:**
- `__init__(self, rules: list[tuple[str, callable]])` — each rule is `(field_name, scoring_fn)`. `scoring_fn` takes a field value and returns a float score. Missing fields contribute 0.
- `score(self, record: dict) -> float` — sum of all rule scores for this record
- `score_batch(self, records: list[dict]) -> list[float]` — scores for all records

**`CurriculumScheduler`:**
- `__init__(self, records: list[dict], scorer: DifficultyScorer, start_percentile: float = 0.2)`
- Internally, score all records and sort by difficulty
- `get_available_indices(self, progress: float) -> list[int]` — return indices of records whose difficulty score is at or below the max_percentile(progress) cutoff. Uses the original indices (position in the input list).
- `sample_batch(self, batch_size: int, progress: float) -> list[int]` — randomly sample batch_size indices from available records at this progress. If batch_size > available count, sample with replacement.
- `get_difficulty_stats(self) -> dict` — return `{"min": float, "max": float, "mean": float, "median": float, "scores": list[float]}` where scores is in original record order

In [ ]:
class DifficultyScorer:
    # YOUR CODE HERE
    pass


class CurriculumScheduler:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 4 ---
torch.manual_seed(42)
np.random.seed(42)

# === DifficultyScorer ===
scorer = DifficultyScorer([
    ("caption_length", lambda x: x / 100.0),  # longer caption = harder
    ("resolution", lambda x: x / 1024.0),      # higher res = harder
])

# Basic scoring
easy_record = {"caption_length": 10, "resolution": 128}
hard_record = {"caption_length": 90, "resolution": 1024}
score_easy = scorer.score(easy_record)
score_hard = scorer.score(hard_record)
assert score_easy < score_hard, f"Easy ({score_easy}) should be less than hard ({score_hard})"

# Missing field contributes 0
partial_record = {"caption_length": 50}
score_partial = scorer.score(partial_record)
assert abs(score_partial - 50/100.0) < 1e-5, f"Expected 0.5, got {score_partial}"

# Batch scoring
batch_scores = scorer.score_batch([easy_record, hard_record, partial_record])
assert len(batch_scores) == 3
assert abs(batch_scores[0] - score_easy) < 1e-5
assert abs(batch_scores[1] - score_hard) < 1e-5

# === CurriculumScheduler ===
# Create records with clear difficulty tiers
records = []
for i in range(100):
    records.append({
        "caption_length": i,          # 0 to 99
        "resolution": 128 + i * 9,    # 128 to 1019
    })

scheduler = CurriculumScheduler(records, scorer, start_percentile=0.2)

# At progress=0.0, only easiest 20% should be available
avail_start = scheduler.get_available_indices(progress=0.0)
assert len(avail_start) == 20, f"Expected 20 available at start, got {len(avail_start)}"
# These should be the records with the lowest difficulty scores (indices 0-19)
assert set(avail_start) == set(range(20)), f"Expected indices 0-19, got {sorted(avail_start)}"

# At progress=0.5, should have 60% available
# max_percentile = min(1.0, 0.2 + 0.5 * 0.8) = min(1.0, 0.6) = 0.6
avail_mid = scheduler.get_available_indices(progress=0.5)
assert len(avail_mid) == 60, f"Expected 60 available at p=0.5, got {len(avail_mid)}"

# At progress=1.0, everything should be available
avail_end = scheduler.get_available_indices(progress=1.0)
assert len(avail_end) == 100, f"Expected 100 available at end, got {len(avail_end)}"

# Monotonicity: more progress → more data
for p in [0.0, 0.25, 0.5, 0.75, 1.0]:
    avail = scheduler.get_available_indices(p)
    if p > 0:
        assert len(avail) >= prev_count, "Available data should increase with progress"
    prev_count = len(avail)

# sample_batch at progress=0 should only return easy items
batch_early = scheduler.sample_batch(batch_size=10, progress=0.0)
assert len(batch_early) == 10
assert all(idx < 20 for idx in batch_early), f"Early batch should only have easy items: {batch_early}"

# sample_batch with replacement when batch_size > available
big_batch = scheduler.sample_batch(batch_size=50, progress=0.0)
assert len(big_batch) == 50  # only 20 available, so must sample with replacement
assert all(idx < 20 for idx in big_batch)

# Difficulty stats
stats = scheduler.get_difficulty_stats()
assert stats["min"] < stats["max"]
assert abs(stats["mean"] - np.mean(stats["scores"])) < 1e-5
assert len(stats["scores"]) == 100
# First record (easiest) should have lowest score
assert stats["scores"][0] < stats["scores"][99]

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Streaming Aggregator with Approximate Statistics

### Scenario

You're processing a large dataset that doesn't fit in memory. Build a streaming aggregator that computes statistics in a single pass, using bounded memory.

### Formulas

**Welford's online algorithm** for running mean and variance:
```
Initialize: count=0, mean=0, M2=0
For each new value x:
    count += 1
    delta = x - mean
    mean += delta / count
    delta2 = x - mean
    M2 += delta * delta2
Variance = M2 / count  (population variance)
```

**Bounded frequent-item tracking (Space-Saving algorithm, simplified):**
- Maintain a Counter with at most `max_tracked` entries
- For each new value: if it's in the counter, increment. If not and counter is not full, add with count 1. If counter is full, find the entry with the minimum count, replace it with the new value at that count + 1.
- This guarantees that any value with frequency > N/max_tracked will be tracked.

**Simplified cardinality estimation (LogLog-style):**
- Hash each value to an integer
- Count the number of leading zeros in the binary representation of the hash
- Track the maximum number of leading zeros seen (`max_zeros`)
- Estimated cardinality ≈ `2^(max_zeros + 0.5)` (the +0.5 is a bias correction)
- Use Python's built-in `hash()` function

### Requirements

**`StreamingAggregator`:**
- `__init__(self, tracked_fields: list[str], max_frequent: int = 100)`
  - `tracked_fields`: which numeric fields to track mean/variance for
  - `max_frequent`: max entries in the frequent-value tracker per field
- `process(self, record: dict)` — process a single record. Each record is a flat dict of field→value.
- `report(self) -> dict`:
  - `"count"`: total records processed
  - `"field_stats"`: dict mapping each tracked field to `{"mean": float, "variance": float}`
  - `"frequent_values"`: dict mapping each string field to list of `(value, count)` sorted by count descending (top items from the bounded tracker)
  - `"cardinality_estimates"`: dict mapping each string field to estimated unique count

A "string field" is any field whose first observed value is a string. A "numeric field" (tracked field) is one listed in `tracked_fields`.

**Memory bound:** The aggregator's internal state should not grow with the number of records processed. It should be O(tracked_fields * 1 + string_fields * max_frequent).

In [ ]:
class StreamingAggregator:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---
np.random.seed(42)

# === Basic mean/variance with Welford's ===
agg = StreamingAggregator(tracked_fields=["score", "length"], max_frequent=10)

scores = np.random.randn(1000).tolist()
lengths = np.random.randint(1, 100, 1000).astype(float).tolist()
categories = np.random.choice(["cat", "dog", "bird", "fish"], 1000).tolist()

for i in range(1000):
    agg.process({"score": scores[i], "length": lengths[i], "category": categories[i]})

report = agg.report()
assert report["count"] == 1000

# Mean should be close to numpy
assert abs(report["field_stats"]["score"]["mean"] - np.mean(scores)) < 1e-6, \
    f"Score mean: {report['field_stats']['score']['mean']} vs {np.mean(scores)}"
assert abs(report["field_stats"]["length"]["mean"] - np.mean(lengths)) < 1e-6

# Variance should be close to numpy (population variance)
assert abs(report["field_stats"]["score"]["variance"] - np.var(scores)) < 1e-4, \
    f"Score var: {report['field_stats']['score']['variance']} vs {np.var(scores)}"
assert abs(report["field_stats"]["length"]["variance"] - np.var(lengths)) < 1e-2

# === Frequent values ===
assert "category" in report["frequent_values"]
freq_cats = report["frequent_values"]["category"]
# All 4 categories should be tracked (max_frequent=10, only 4 unique values)
tracked_labels = {v for v, c in freq_cats}
assert tracked_labels == {"cat", "dog", "bird", "fish"}, f"Expected all 4 categories, got {tracked_labels}"
# Should be sorted by count descending
counts = [c for v, c in freq_cats]
assert counts == sorted(counts, reverse=True), "Should be sorted by count desc"
# Total should be close to 1000 (exact if no eviction)
total_tracked = sum(c for v, c in freq_cats)
assert total_tracked == 1000, f"Total tracked should be 1000, got {total_tracked}"

# === Cardinality estimation ===
assert "category" in report["cardinality_estimates"]
est_cardinality = report["cardinality_estimates"]["category"]
# True cardinality is 4. Estimate should be within 50% → between 2 and 6
assert 2 <= est_cardinality <= 8, f"Cardinality estimate {est_cardinality} too far from 4"

# === Bounded memory test ===
# Process a stream with many unique string values, but bounded tracker
agg2 = StreamingAggregator(tracked_fields=["val"], max_frequent=5)
for i in range(10000):
    agg2.process({"val": float(i), "tag": f"item_{i % 100}"})

report2 = agg2.report()
# Frequent values should have at most max_frequent entries
assert len(report2["frequent_values"]["tag"]) <= 5, \
    f"Frequent tracker should be bounded at 5, got {len(report2['frequent_values']['tag'])}"

# Cardinality estimate for 100 unique tags should be in the right ballpark
est_card_tags = report2["cardinality_estimates"]["tag"]
assert 50 <= est_card_tags <= 200, f"Cardinality estimate {est_card_tags} too far from 100"

# Mean of val (0..9999) should be ~4999.5
assert abs(report2["field_stats"]["val"]["mean"] - 4999.5) < 1e-3

# === Edge case: single record ===
agg3 = StreamingAggregator(tracked_fields=["x"], max_frequent=5)
agg3.process({"x": 42.0})
r3 = agg3.report()
assert r3["count"] == 1
assert abs(r3["field_stats"]["x"]["mean"] - 42.0) < 1e-5
assert abs(r3["field_stats"]["x"]["variance"] - 0.0) < 1e-5

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Mini Model Evaluator

### Scenario

Build a complete model evaluation pipeline that computes classification metrics, calibration statistics, and model comparison with statistical significance.

### Formulas

**Precision/Recall/F1 (per class c):**
```
TP_c = items where predicted=c AND actual=c
FP_c = items where predicted=c AND actual!=c
FN_c = items where predicted!=c AND actual=c
precision_c = TP_c / (TP_c + FP_c)    [0.0 if denominator is 0]
recall_c = TP_c / (TP_c + FN_c)       [0.0 if denominator is 0]
f1_c = 2 * precision_c * recall_c / (precision_c + recall_c)  [0.0 if denominator is 0]
```

**Macro F1:** arithmetic mean of per-class F1 scores

**Weighted F1:** weighted mean of per-class F1, weighted by class support (number of actual items in that class)
```
weighted_f1 = sum(f1_c * support_c for c in classes) / sum(support_c for c in classes)
```

**Confusion matrix:** `confusion[actual][predicted]` = count

**Calibration:** Divide predictions into confidence buckets [0.0-0.1), [0.1-0.2), ..., [0.9-1.0]. For each bucket:
- `avg_confidence`: mean confidence of predictions in this bucket
- `avg_accuracy`: fraction of predictions in this bucket that were correct
- `count`: number of predictions in this bucket

A well-calibrated model has `avg_confidence ≈ avg_accuracy` in each bucket.

**Permutation test for comparing two models (p-value):**
1. Compute observed F1 gap = |F1_model_A - F1_model_B|
2. For N=1000 shuffles:
   a. For each prediction, randomly swap which model gets it (flip a fair coin per item)
   b. Compute F1 for each shuffled "model"
   c. Record the F1 gap for this shuffle
3. p-value = (number of shuffles where gap >= observed gap) / N

### Requirements

**`evaluate(predictions: list[tuple[str, str, float]], ground_truth: dict[str, str]) -> dict`:**
- `predictions`: list of (item_id, predicted_label, confidence)
- `ground_truth`: dict mapping item_id → true_label
- Return:
  - `"per_class"`: dict mapping class → `{"precision": float, "recall": float, "f1": float, "support": int}`
  - `"macro_f1"`: float
  - `"weighted_f1"`: float
  - `"confusion_matrix"`: dict of dict, `confusion[actual][predicted]` = count
  - `"calibration"`: list of `{"bucket": str, "avg_confidence": float, "avg_accuracy": float, "count": int}` for each non-empty bucket

**`compare_models(model_predictions: dict[str, list[tuple[str, str, float]]], ground_truth: dict[str, str], n_permutations: int = 1000) -> dict`:**
- `model_predictions`: dict mapping model_name → predictions list
- Evaluate each model
- Rank by macro F1 descending
- Compute p-value between top model and second model using permutation test
- Return:
  - `"rankings"`: list of `{"model_name": str, "macro_f1": float}` sorted by F1 desc
  - `"evaluations"`: dict mapping model_name → full evaluation dict
  - `"significance"`: `{"top": str, "second": str, "p_value": float}` (omit if only one model)

In [ ]:
def evaluate(predictions: list[tuple[str, str, float]], ground_truth: dict[str, str]) -> dict:
    # YOUR CODE HERE
    pass


def compare_models(
    model_predictions: dict[str, list[tuple[str, str, float]]],
    ground_truth: dict[str, str],
    n_permutations: int = 1000,
) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 6 ---
np.random.seed(42)

# === Perfect model ===
gt = {"a": "cat", "b": "dog", "c": "cat", "d": "bird", "e": "dog"}
perfect_preds = [(k, v, 1.0) for k, v in gt.items()]

result = evaluate(perfect_preds, gt)
assert abs(result["macro_f1"] - 1.0) < 1e-5, f"Perfect model should have F1=1.0, got {result['macro_f1']}"
assert abs(result["weighted_f1"] - 1.0) < 1e-5
for cls, metrics in result["per_class"].items():
    assert abs(metrics["precision"] - 1.0) < 1e-5
    assert abs(metrics["recall"] - 1.0) < 1e-5
    assert abs(metrics["f1"] - 1.0) < 1e-5

# === Confusion matrix ===
assert result["confusion_matrix"]["cat"]["cat"] == 2
assert result["confusion_matrix"]["dog"]["dog"] == 2
assert result["confusion_matrix"]["bird"]["bird"] == 1

# === Imperfect model ===
gt2 = {f"item_{i}": ["cat", "dog", "bird"][i % 3] for i in range(30)}
# Model that confuses cat and dog sometimes
imperfect_preds = []
for item_id, true_label in gt2.items():
    idx = int(item_id.split("_")[1])
    if true_label == "cat" and idx % 5 == 0:
        pred_label = "dog"  # misclassify some cats as dogs
        conf = 0.4
    elif true_label == "dog" and idx % 7 == 0:
        pred_label = "bird"  # misclassify some dogs as birds
        conf = 0.3
    else:
        pred_label = true_label
        conf = 0.85 + 0.1 * np.random.random()
    imperfect_preds.append((item_id, pred_label, conf))

result2 = evaluate(imperfect_preds, gt2)
assert 0 < result2["macro_f1"] < 1.0, f"Imperfect model F1 should be between 0 and 1: {result2['macro_f1']}"
assert 0 < result2["weighted_f1"] < 1.0

# Per-class support should match
for cls, metrics in result2["per_class"].items():
    assert metrics["support"] == 10, f"Each class should have 10 items, {cls} has {metrics['support']}"
    assert 0 <= metrics["precision"] <= 1.0
    assert 0 <= metrics["recall"] <= 1.0

# === Calibration ===
calibration = result2["calibration"]
assert len(calibration) > 0
for bucket in calibration:
    assert "bucket" in bucket
    assert "avg_confidence" in bucket
    assert "avg_accuracy" in bucket
    assert "count" in bucket
    assert bucket["count"] > 0
    assert 0 <= bucket["avg_confidence"] <= 1.0
    assert 0 <= bucket["avg_accuracy"] <= 1.0

# === Edge case: class with zero predictions ===
gt3 = {"x": "cat", "y": "cat", "z": "dog"}
preds3 = [("x", "cat", 0.9), ("y", "cat", 0.8), ("z", "cat", 0.7)]  # never predicts "dog"
result3 = evaluate(preds3, gt3)
# "dog" recall should be 0 (never predicted correctly)
assert result3["per_class"]["dog"]["recall"] == 0.0
# "cat" precision should be 2/3 (predicted cat 3 times, correct 2 times)
assert abs(result3["per_class"]["cat"]["precision"] - 2/3) < 1e-5

# === compare_models ===
np.random.seed(42)
gt_compare = {f"i{i}": ["A", "B"][i % 2] for i in range(100)}

# Good model: 90% accuracy
good_preds = []
for item_id, true_label in gt_compare.items():
    if np.random.random() < 0.9:
        good_preds.append((item_id, true_label, 0.9))
    else:
        other = "B" if true_label == "A" else "A"
        good_preds.append((item_id, other, 0.4))

# Bad model: 60% accuracy
np.random.seed(123)
bad_preds = []
for item_id, true_label in gt_compare.items():
    if np.random.random() < 0.6:
        bad_preds.append((item_id, true_label, 0.7))
    else:
        other = "B" if true_label == "A" else "A"
        bad_preds.append((item_id, other, 0.5))

np.random.seed(42)
comparison = compare_models(
    {"good_model": good_preds, "bad_model": bad_preds},
    gt_compare,
    n_permutations=500
)

# Rankings
assert len(comparison["rankings"]) == 2
assert comparison["rankings"][0]["model_name"] == "good_model", \
    f"Good model should rank first, got {comparison['rankings'][0]['model_name']}"
assert comparison["rankings"][0]["macro_f1"] > comparison["rankings"][1]["macro_f1"]

# Evaluations
assert "good_model" in comparison["evaluations"]
assert "bad_model" in comparison["evaluations"]

# Significance: p-value should be small (models are clearly different)
assert "significance" in comparison
assert comparison["significance"]["top"] == "good_model"
assert comparison["significance"]["second"] == "bad_model"
assert 0 <= comparison["significance"]["p_value"] <= 1.0
# With a 90% vs 60% accuracy gap on 100 items, p-value should be very small
assert comparison["significance"]["p_value"] < 0.05, \
    f"P-value should be < 0.05 for clearly different models, got {comparison['significance']['p_value']}"

# === Single model comparison (no significance) ===
single = compare_models({"only": good_preds}, gt_compare, n_permutations=100)
assert len(single["rankings"]) == 1
assert "significance" not in single or single.get("significance") is None

print("Problem 6: ALL TESTS PASSED")

---

## Scoring

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Overqualified** — you're ready for anything |
| Solved in 25-30 min, minor debugging | **Strong pass** — these are at the ceiling of CoderPad difficulty |
| 30-40 min or needed hints | **Pass** — you can handle hard problems, keep practicing |
| Couldn't solve or > 40 min | **Go back to HARD** — nail those first, then return here |